### Imports and Data Loading

In [1]:
import os
import ast
from pathlib import Path
import pandas as pd
import string
import nltk
import contractions
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize tqdm for pandas tracking
tqdm.pandas()

# File paths (all CSV now)
DATA_PATH = Path("rotten_tomatoes_critic_reviews.csv")
PREPROCESSED_PATH = Path("preprocessed_full_dataset.csv")
M3_OUTPUT_PATH = Path("FINAL_BALANCED_FOR_M3.csv")
M2_TRAIN_PATH = Path("tfidf_train.csv")
M2_TEST_PATH = Path("tfidf_test.csv")
CSV_OUTPUT_PATH = Path("FINAL_BALANCED_FOR_M2_M3.csv")

# Download required NLTK resources only if missing
def ensure_nltk_resource(resource_path, download_name):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(download_name)

ensure_nltk_resource("tokenizers/punkt", "punkt")
ensure_nltk_resource("tokenizers/punkt_tab", "punkt_tab")
ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")
ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")
ensure_nltk_resource("taggers/averaged_perceptron_tagger", "averaged_perceptron_tagger")
ensure_nltk_resource("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng")

print("Loading full dataset...")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Put the CSV in the same folder as this notebook.")

raw_df = pd.read_csv(DATA_PATH)
raw_df.columns = raw_df.columns.str.strip()

required_columns = {"review_type", "review_content"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise KeyError(f"Missing required column(s): {missing_columns}. Available columns: {list(raw_df.columns)}")

print(f"Raw dataset rows: {len(raw_df):,}")

# Keep only rows with valid review content and sentiment label
full_df = raw_df.dropna(subset=["review_type", "review_content"]).copy()
full_df["review_type"] = full_df["review_type"].astype(str).str.strip()
full_df["review_content"] = full_df["review_content"].astype(str).str.strip()

# Remove empty review text after stripping spaces
full_df = full_df[full_df["review_content"].ne("")].copy()

# Keep only the two sentiment classes used in this project
full_df = full_df[full_df["review_type"].isin(["Fresh", "Rotten"])].copy()
print(f"Valid Fresh/Rotten reviews before duplicate removal: {len(full_df):,}")

# Remove duplicate reviews based on normalized review_content while keeping all columns
before_dupes = len(full_df)
full_df["review_content_normalized"] = (
    full_df["review_content"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

full_df = full_df.drop_duplicates(subset=["review_content_normalized"], keep="first").copy()
removed_dupes = before_dupes - len(full_df)

# Remove temporary helper column and reset index for clean downstream processing
full_df = full_df.drop(columns=["review_content_normalized"]).reset_index(drop=True)

print(f"Duplicate review_content rows removed: {removed_dupes:,}")
print(f"Remaining reviews in full_df: {len(full_df):,}")
print("\nClass distribution after duplicate removal:")
print(full_df["review_type"].value_counts())

# Safety check so later cells do not fail on review_type
assert "review_type" in full_df.columns, "review_type column is missing after duplicate removal."
assert "review_content" in full_df.columns, "review_content column is missing after duplicate removal."


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Loading full dataset...
Raw dataset rows: 1,130,017
Valid Fresh/Rotten reviews before duplicate removal: 1,064,211
Duplicate review_content rows removed: 115,095
Remaining reviews in full_df: 949,116

Class distribution after duplicate removal:
review_type
Fresh     607303
Rotten    341813
Name: count, dtype: int64


### Preprocessing Pipeline (10 Steps)

In [2]:
# Setup NLP tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print(f"Starting preprocessing on {len(full_df):,} rows...")

# 1. Sentence Segmentation
full_df["sentences"] = full_df["review_content"].progress_apply(sent_tokenize)
affected_sent = full_df[full_df["sentences"].apply(len) > 1].shape[0]

# 2. Case Folding
full_df["lower"] = full_df["review_content"].str.lower()
affected_lower = (full_df["review_content"] != full_df["lower"]).sum()

# 3. Abbreviation / Contraction Expansion
def fix_contractions_safe(text):
    if contractions is None:
        return text
    return contractions.fix(text)

full_df["expanded"] = full_df["lower"].progress_apply(fix_contractions_safe)
affected_abbr = (full_df["lower"] != full_df["expanded"]).sum()

# 4. Word Tokenization
full_df["tokens"] = full_df["expanded"].progress_apply(word_tokenize)

# 5. POS Tagging
print("Running POS tagging on tokenized reviews...")
full_df["pos_tags"] = full_df["tokens"].progress_apply(pos_tag)

# 6. Punctuation Removal
full_df["no_punc"] = full_df["expanded"].progress_apply(
    lambda x: x.translate(str.maketrans("", "", string.punctuation))
)
affected_punc = (full_df["expanded"] != full_df["no_punc"]).sum()

# 7. ML Tokenization after punctuation removal
full_df["ml_tokens"] = full_df["no_punc"].progress_apply(word_tokenize)

# 8. Stopword Removal
full_df["no_stop"] = full_df["ml_tokens"].progress_apply(
    lambda tokens: [w for w in tokens if w not in stop_words]
)
affected_stop = (full_df["ml_tokens"].apply(len) != full_df["no_stop"].apply(len)).sum()

# 9. Lemmatization with POS support
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_with_pos(row):
    pos_dict = {word: get_wordnet_pos(tag) for word, tag in row["pos_tags"]}
    return [lemmatizer.lemmatize(word, pos_dict.get(word, wordnet.NOUN)) for word in row["no_stop"]]

full_df["lemmatized"] = full_df.progress_apply(lemmatize_with_pos, axis=1)

affected_lemma = (
    full_df["no_stop"].apply(lambda x: " ".join(x)) !=
    full_df["lemmatized"].apply(lambda x: " ".join(x))
).sum()

# 10. Non-Alphabetic Filtering
full_df["alpha_only"] = full_df["lemmatized"].progress_apply(
    lambda tokens: [w for w in tokens if w.isalpha()]
)
affected_alpha = (full_df["lemmatized"].apply(len) != full_df["alpha_only"].apply(len)).sum()

# 11. Final Cleaned String Generation for Traditional ML Models
full_df["final_cleaned"] = full_df["alpha_only"].apply(lambda tokens: " ".join(tokens))

# Remove rows that became empty after cleaning
before_empty_cleaned = len(full_df)
full_df = full_df[full_df["final_cleaned"].str.strip().ne("")].copy().reset_index(drop=True)
removed_empty_cleaned = before_empty_cleaned - len(full_df)

# Summary report
print("\nPREPROCESSING REPORT (FULL DATASET)")
print(f"Total Rows Processed After Cleaning: {len(full_df):,}")
print(f"1. Rows with Multiple Sentences: {affected_sent:,}")
print(f"2. Rows with Casing Changes: {affected_lower:,}")
print(f"3. Rows with Contractions Fixed: {affected_abbr:,}")
print(f"4. Rows with Punctuation Removed: {affected_punc:,}")
print(f"5. Rows with Stopwords Removed: {affected_stop:,}")
print(f"6. Rows Lemmatized: {affected_lemma:,}")
print(f"7. Rows with Non-Alpha Removed: {affected_alpha:,}")
print(f"8. Empty cleaned rows removed: {removed_empty_cleaned:,}")

# Serialize list columns to strings so they survive CSV round-trips
# pos_tags: list of (word, tag) tuples  ->  str
# lemmatized: list of str tokens        ->  str
save_df = full_df.copy()
save_df["pos_tags"] = save_df["pos_tags"].apply(str)
save_df["lemmatized"] = save_df["lemmatized"].apply(str)

# Keep only the columns needed downstream to keep file size manageable
save_cols = ["review_type", "review_content", "pos_tags", "lemmatized", "final_cleaned"]
save_df[save_cols].to_csv(PREPROCESSED_PATH, index=False)
print(f"\nSaved regenerated preprocessing file: {PREPROCESSED_PATH}")


Starting preprocessing on 949,116 rows...


100%|██████████| 949116/949116 [05:16<00:00, 2995.80it/s]


Running POS tagging on tokenized reviews...


100%|██████████| 949116/949116 [00:04<00:00, 194998.87it/s]



PREPROCESSING REPORT (FULL DATASET)
Total Rows Processed After Cleaning: 949,050
1. Rows with Multiple Sentences: 189,635
2. Rows with Casing Changes: 931,446
3. Rows with Contractions Fixed: 276,642
4. Rows with Punctuation Removed: 940,515
5. Rows with Stopwords Removed: 938,576
6. Rows Lemmatized: 821,603
7. Rows with Non-Alpha Removed: 65,454
8. Empty cleaned rows removed: 66

Saved regenerated preprocessing file: preprocessed_full_dataset.csv


### Data Balancing and Clean Handoff Exports

In [3]:
print("Preparing balanced dataset...")

# Helper to safely parse stringified list columns back to Python objects
def parse_list_col(series):
    return series.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Use the current notebook variable if available. If not, load the regenerated CSV.
if "full_df" in globals() and {"review_type", "review_content", "pos_tags", "lemmatized", "final_cleaned"}.issubset(full_df.columns):
    df_preprocessed = full_df.copy()
    print("Using preprocessed full_df from the current notebook session.")
elif PREPROCESSED_PATH.exists():
    df_preprocessed = pd.read_csv(PREPROCESSED_PATH)
    # Restore list columns from their stringified form
    df_preprocessed["pos_tags"] = parse_list_col(df_preprocessed["pos_tags"])
    df_preprocessed["lemmatized"] = parse_list_col(df_preprocessed["lemmatized"])
    print(f"Loaded {PREPROCESSED_PATH}.")
else:
    raise FileNotFoundError(
        f"{PREPROCESSED_PATH} was not found. Run the preprocessing cell first to regenerate it."
    )

required_for_export = {"review_type", "review_content", "pos_tags", "lemmatized", "final_cleaned"}
missing_for_export = required_for_export - set(df_preprocessed.columns)
if missing_for_export:
    raise KeyError(
        f"Missing required column(s) before balancing: {missing_for_export}. "
        f"Available columns: {list(df_preprocessed.columns)}"
    )

# Keep only valid Fresh/Rotten rows and non-empty cleaned text
df_preprocessed = df_preprocessed.dropna(subset=["review_type", "review_content", "final_cleaned"]).copy()
df_preprocessed["review_type"] = df_preprocessed["review_type"].astype(str).str.strip()
df_preprocessed["final_cleaned"] = df_preprocessed["final_cleaned"].astype(str).str.strip()
df_preprocessed = df_preprocessed[
    df_preprocessed["review_type"].isin(["Fresh", "Rotten"]) &
    df_preprocessed["final_cleaned"].ne("")
].copy()

# Identify minority size for 50/50 balancing
class_counts = df_preprocessed["review_type"].value_counts()
print("\nClass distribution before balancing:")
print(class_counts)

if class_counts.shape[0] < 2:
    raise ValueError("Both Fresh and Rotten classes are required for 50/50 balancing.")

minority_size = class_counts.min()
print(f"\nTargeting {minority_size:,} rows per class for a perfect 50/50 split.")

# Under-sample to balance classes based on minority class size
fresh_df = df_preprocessed[df_preprocessed["review_type"] == "Fresh"].sample(n=minority_size, random_state=42)
rotten_df = df_preprocessed[df_preprocessed["review_type"] == "Rotten"].sample(n=minority_size, random_state=42)
balanced_df = pd.concat([fresh_df, rotten_df], axis=0)

# Random shuffle to mix Fresh and Rotten rows
print("Shuffling final balanced dataset...")
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nClass distribution after balancing:")
print(balanced_df["review_type"].value_counts())

# EXPORT 1: Dataset for Transformer / BERT / Aspect-Based Sentiment Analysis
# pos_tags and lemmatized are Python lists — stringify them for CSV storage
m3_df = balanced_df[["review_type", "review_content", "pos_tags", "lemmatized"]].copy()
m3_df["pos_tags"] = m3_df["pos_tags"].apply(
    lambda x: str(x) if not isinstance(x, str) else x
)
m3_df["lemmatized"] = m3_df["lemmatized"].apply(
    lambda x: str(x) if not isinstance(x, str) else x
)
m3_df.to_csv(M3_OUTPUT_PATH, index=False)
print(f"Saved {M3_OUTPUT_PATH} successfully.")

# EXPORT 2: Train / Test split CSVs for Traditional ML Models
print("\nExecuting train/test split for M2...")

train_df, test_df = train_test_split(
    balanced_df[["review_type", "review_content", "final_cleaned"]],
    test_size=0.20,
    random_state=42,
    stratify=balanced_df["review_type"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(M2_TRAIN_PATH, index=False)
test_df.to_csv(M2_TEST_PATH, index=False)

print(f"Saved {M2_TRAIN_PATH} ({len(train_df):,} rows) successfully.")
print(f"Saved {M2_TEST_PATH} ({len(test_df):,} rows) successfully.")
print("\nPipeline complete. All output files have been regenerated.")


Preparing balanced dataset...
Using preprocessed full_df from the current notebook session.

Class distribution before balancing:
review_type
Fresh     607284
Rotten    341766
Name: count, dtype: int64

Targeting 341,766 rows per class for a perfect 50/50 split.
Shuffling final balanced dataset...

Class distribution after balancing:
review_type
Rotten    341766
Fresh     341766
Name: count, dtype: int64
Saved FINAL_BALANCED_FOR_M3.csv successfully.

Executing train/test split for M2...
Saved tfidf_train.csv (546,825 rows) successfully.
Saved tfidf_test.csv (136,707 rows) successfully.

Pipeline complete. All output files have been regenerated.


In [4]:
# Final CSV export for checking / report evidence
if "balanced_df" not in globals():
    raise NameError("balanced_df is not available. Run the balancing and export cell first.")

final_columns = ["review_type", "review_content", "pos_tags", "lemmatized", "final_cleaned"]
missing_final_columns = set(final_columns) - set(balanced_df.columns)
if missing_final_columns:
    raise KeyError(f"Missing column(s) for final CSV export: {missing_final_columns}")

# Stringify list columns before writing
final_df = balanced_df[final_columns].copy()
final_df["pos_tags"] = final_df["pos_tags"].apply(
    lambda x: str(x) if not isinstance(x, str) else x
)
final_df["lemmatized"] = final_df["lemmatized"].apply(
    lambda x: str(x) if not isinstance(x, str) else x
)
final_df.to_csv(CSV_OUTPUT_PATH, index=False)
print(f"Saved {CSV_OUTPUT_PATH} successfully.")
print(f"Final CSV rows: {len(final_df):,}")


Saved FINAL_BALANCED_FOR_M2_M3.csv successfully.
Final CSV rows: 683,532
